# Week 11 Independent Lab: File System Forensics & Scheduling Decisions

This take-home notebook extends the class lab. Move slowly, read the hints, and capture notes for your writing log at the end.

## What You Will Practice
- Reading an operating system log to understand how applications touch storage.
- Comparing sequential and direct access patterns with small calculations.
- Recommending a disk scheduling strategy for a larger queue.

🛠️ **Tools used:** Python, pandas (for lightweight data analysis), and the same scheduling logic from class.

## Part 1 · Log Sleuth
Imagine a research laptop captured the following file activity during a 10-minute sprint. Each row records:
- `timestamp`: when the event happened.
- `operation`: OPEN, READ, WRITE, DELETE.
- `path`: full file path (directory + filename).
- `bytes`: payload size, or 0 for metadata-only actions.
- `access`: the pattern the OS detected (sequential vs. direct).

You will answer investigative questions about this log.

### Guided Tasks
1. Load the log into a pandas DataFrame (the code below does this for you).
2. Compute:
   - Total bytes **read** vs. **written**.
   - Top three file extensions by number of operations.
   - Maximum folder depth (hint: split the path on `/`).
3. Flag risky operations, such as executables in `Downloads` or leftover temp files.

> **Hint:** `Path(path_string).suffix` and `path_string.count('/')` will be helpful.

In [ ]:
from collections import Counter
from pathlib import Path
import pandas as pd

LOG_ENTRIES = [
    {'timestamp': '2024-04-02T09:01:12', 'operation': 'OPEN', 'path': '/Users/student/Documents/research/notes.txt', 'bytes': 4096, 'access': 'sequential'},
    {'timestamp': '2024-04-02T09:01:14', 'operation': 'READ', 'path': '/Users/student/Documents/research/data.csv', 'bytes': 524288, 'access': 'sequential'},
    {'timestamp': '2024-04-02T09:02:05', 'operation': 'WRITE', 'path': '/Users/student/Documents/research/results.db', 'bytes': 262144, 'access': 'direct'},
    {'timestamp': '2024-04-02T09:03:22', 'operation': 'OPEN', 'path': '/Users/student/Desktop/screenshots/final.png', 'bytes': 1048576, 'access': 'direct'},
    {'timestamp': '2024-04-02T09:04:18', 'operation': 'READ', 'path': '/Users/student/Documents/research/data.csv', 'bytes': 524288, 'access': 'sequential'},
    {'timestamp': '2024-04-02T09:05:09', 'operation': 'WRITE', 'path': '/Users/student/Documents/research/cache.tmp', 'bytes': 32768, 'access': 'direct'},
    {'timestamp': '2024-04-02T09:05:45', 'operation': 'DELETE', 'path': '/Users/student/Downloads/installer.dmg', 'bytes': 0, 'access': 'direct'},
    {'timestamp': '2024-04-02T09:07:10', 'operation': 'READ', 'path': '/Users/student/Documents/research/index.idx', 'bytes': 65536, 'access': 'direct'},
    {'timestamp': '2024-04-02T09:08:33', 'operation': 'OPEN', 'path': '/Users/student/Documents/essays/draft.docx', 'bytes': 175000, 'access': 'sequential'},
    {'timestamp': '2024-04-02T09:09:57', 'operation': 'WRITE', 'path': '/Users/student/Documents/research/checkpoints.chk', 'bytes': 8192, 'access': 'direct'},
]

df = pd.DataFrame(LOG_ENTRIES)
df


In [ ]:
# TODO: Calculate total bytes read and written.
# Hint: use DataFrame filters such as df[df['operation'] == 'READ']['bytes'].sum()


In [ ]:
# TODO: Find the top three extensions by operation count.
# Consider using Counter or pandas value_counts on a derived 'extension' column.


In [ ]:
# TODO: Compute the maximum directory depth (number of folders) seen in the log.
# Example: '/Users/student/Documents/research/data.csv' has depth 5 (Users, student, Documents, research, file).


### Risk Report
List any log entries you would investigate further. Explain why in a short bullet list (e.g., “`installer.dmg` in Downloads was deleted—check if it was malicious or failed install”). Use Python or plain text—your choice.

## Part 2 · Sequential vs. Direct Snapshot
Using the log above:
1. Compute the **average bytes per operation** for sequential vs. direct accesses.
2. Describe (2–3 sentences) what this tells you about the workload. Are sequential tasks larger? Which files drive the direct pattern?
3. Imagine this workload ran every hour. How might OS caching change performance over time?

In [ ]:
# TODO: Calculate the average bytes per operation for each access type.


📝 **Write here:** Summarise your comparison and caching hypothesis.

## Part 3 · Scheduling at Scale
A shared storage server receives the queue below. The head starts at track 215 and was previously moving upward.

```
Queue: [215, 30, 489, 176, 320, 241, 90, 17, 384, 276, 125, 60]
```

### Tasks
1. Reuse your in-class functions (or copy them below) for FCFS, SSTF, and C-SCAN.
2. Record the service order, total head movement, and average wait time for each.
3. Recommend a policy and support your choice with at least two pieces of evidence.
4. ⭐ Bonus: predict what changes if the drive is an SSD (seek cost ≈ 0).

> **Reminder:** C-SCAN sweeps upward, then *wraps* to the start of the disk (track 0) without serving requests on the way down.

In [ ]:
def fcfs(start, requests):
    order = list(requests)
    movement = 0
    current = start
    waits = []
    for idx, track in enumerate(order):
        movement += abs(track - current)
        waits.append(movement if idx > 0 else abs(track - current))
        current = track
    avg_wait = sum(waits) / len(waits)
    return order, movement, avg_wait

queue = [215, 30, 489, 176, 320, 241, 90, 17, 384, 276, 125, 60]

fcfs_order, fcfs_move, fcfs_wait = fcfs(215, queue)
fcfs_order, fcfs_move, round(fcfs_wait, 2)


In [ ]:
# TODO: Implement SSTF for the home-lab queue (recycle your class code if needed).


In [ ]:
# TODO: Implement C-SCAN (wrap to track 0 after reaching the highest request).


### Recommendation
Write 5–6 sentences comparing your metrics. Mention fairness, total movement, and how the workload characteristics influenced your pick.

## Reflection Notes (for your writing log)
Fill in each item with quick bullet points. You will expand these into paragraphs in the writing log submission.
- **Habit change:**
- **Access insight:**
- **Open question:**
